In [ ]:
import rclpy
from rclpy.node import Node
import gc

from std_msgs.msg import String
from geometry_msgs.msg import Pose
from geographic_msgs.msg import GeoPoint
from lotusim_msgs.msg import VesselCmd, VesselCmdArray, VesselPositionArray

The messages (VesselCmd, VesselCmdArray) are in the interface folder. Please open it. 

In VesselCmd msg, the cmd is in a json string format as LOTUSim aims to be modular and the actuator msg are not catered to any physics engine and meant to be generic

The physics interface developed will then read the `cmd_string` and parse it to fit the physics engine you choose
```
std_msgs/Header header

string vessel_name

uint16 entity

# cmd in json string format
string cmd_string
```

Each variable is treated like a class variable.

Below are example of publisher, subscriber using xdyn as a physics engine.

Xdyn consume propeller command using propeller_name(rpm), propeller_name(P/D), where P/D stands for pitch to diameter ratio. For more information regarding xdyn, read the xdyn developer doc  

In [ ]:
class ExampleNode(Node):
    def __init__(self):
        super().__init__('example_control_node')
        self.pose_subscription = self.create_subscription(
            VesselPositionArray,
            '/lotusim/poses',
            self.poses_callback,
            10)
        self.vessel_poses= {}
        self.cmd_publisher = self.create_publisher(
            VesselCmdArray,
            '/lotusim/vessel_cmd_array'
        )

    def poses_callback(self, msg):
        for vessel_position in msg.vessels:
            lat = vessel_position.geo_point.latitude
            long = vessel_position.geo_point.longitude
            self.vessel_poses[vessel_position.vessel_name] = (lat, long)

    def spawn_ship_with_dynamics(self,id):
        msg = MASCmdMsg()
        msg.cmd_type =  MASCmdMsg.CREATE_CMD
        msg.model_name =  "lrauv"
        msg.vessel_name =  "lrauv_" + str(id)        
        latlong_msg = GeoPoint()
        latlong_msg.latitude = 1.2605794416293148
        latlong_msg.longitude = 103.7516212463379
        latlong_msg.altitude = -10.0
        msg.geo_point = latlong_msg
        
        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>true</publish_render>
                <renderer_type_name>lrauv</renderer_type_name>
            </render_interface>
            <physics_engine_interface>
            <underwater>
                <connection_type>XDynWebSocket</connection_type>
                <uri>ws://127.0.0.1:12346</uri>
                <thrusters>
                    <thrusters1>thruster1</thrusters1>
                </thrusters>
            </underwater>
            <surface>
                <connection_type>XDynWebSocket</connection_type>
                <uri>ws://127.0.0.1:12345</uri>
                <thrusters>
                    <thrusters1>thruster1</thrusters1>
                </thrusters>
            </surface>
            <init_state>Underwater</init_state>
            </physics_engine_interface>
        </lotus_param>
        """

    def send_propeller_command(self, vessel_name:str, propeller_name: str, rpm: float, pd: float):
        """
        Prepare and send a propeller command message.

        Args:
            rpm (float): Propeller rotation speed in RPM.
            pd (float): Propeller pitch-to-diameter ratio.
        """
        # Create a VesselCmdArray message
        cmd_array = VesselCmdArray()
        cmd = VesselCmd()
        cmd.vessel_name = vessel_name 
        cmd.cmd_string = json.dumps({"propeller_name(rpm)": rpm, "propeller_name(P/D)": pd})
        cmd_array.cmds.append(cmd)

        # Publish the message
        self.cmd_publisher.publish(cmd_array)

        # Also log with ROS logger
        printf("{cmd.vessel_name} - propeller command published: rpm={rpm}, P/D={pd}")

def main(args=None):
    if not rclpy.ok(): 
        rclpy.init(args=args)

    node = ExampleNode()
    node.send_mas_array()

    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("Shutting down...")
    finally:
        node.destroy_node()
        rclpy.shutdown()

In [ ]:
if not rclpy.ok(): 
    rclpy.init(args=None)

try:
    stop_executor()  
except NameError:
    pass 
except Exception as e:
    print(f"Could not destroy resources: {e}")

node = ExampleNode()
executor = MultiThreadedExecutor()
executor.add_node(node)

def spin_executor():
    executor.spin()
def stop_executor():
    """Call this function to stop the spinning thread"""
    try:
        executor.shutdown()
        spin_thread.join(timeout=2.0)
        node.destroy_node()
        print("Executor stopped successfully")
    except Exception as e:
        print(f"Error stopping executor: {e}")

spin_thread = threading.Thread(target=spin_executor, daemon=True)
spin_thread.start()
print("Node spinning in background thread")

In [ ]:
spawn_ship_with_dynamics(1)

In [ ]:
send_propeller_command("lrauv_1", "propeller", 200, 0.88)